# Description

This code helps the screening process by cleaning the WoS and Scopus queries results.


# Prerequisites

- pandas
- openpyxl
- xlrd 


Use the following to install the packages:

```pip install -e ".[screening]"```

# Input

- Excel/csv file format in the `data` folder of the scopus/WoS result query (see `filename_scopus` and `filename_wos`)


In [ ]:
import pandas as pd

In [ ]:
filename_scopus = "../data/query_results/scopus_export_May 25-2026_07542f75-52e9-45c0-ab62-cd18e10fb02d.csv"
filename_wos = "../data/query_results/savedrecs.xls"


In [ ]:
scopus = pd.read_csv(filename_scopus)
scopus['Title'] = scopus['Title'].str.lower()

In [ ]:
wos = pd.read_excel(filename_wos)
wos['Article Title'] = wos['Article Title'].str.lower()

In [ ]:
etl_columns = ['Title', 'Authors', 'Year', 'Source', 
               'Abstract','DOI',
               'Author Keywords',
               'Index Keywords', 
               'Language of Original Document',
               'Document Type']

etl_corresponding_columns = {
    'Article Title': 'Title',
    'Publication Year': 'Year',
    'Source Title': 'Source',
    'Language':'Language of Original Document',
    'Keywords Plus':'Index Keywords'
}

In [ ]:
wos.rename(columns=etl_corresponding_columns, inplace=True)

In [ ]:
wos = wos[etl_columns].copy()
scopus = scopus[etl_columns].copy()

In [ ]:
print(f"WoS: {len(wos)}, Scopus: {len(scopus)}")


In [ ]:
df = pd.concat([wos, scopus], ignore_index=True)
len_before = len(df)
print("Before removing duplicates:", len_before)
df = df.drop_duplicates(subset="Title", keep="last").reset_index(drop=True)
len_after = len(df)
print("After removing duplicates:", len_after)

In [ ]:
print("Duplicates removed:", len_before - len_after)

In [ ]:
df = df[df['Language of Original Document'].isin(['English'])]

In [ ]:
len_english = len(df)

In [ ]:
print(f"Non english documents: {len_after - len_english}")

In [ ]:
print(f"Number of documents for screening: {len(df)}")

In [ ]:
df.to_excel("../data/query_results/combined_screened.xlsx")